# 02 · Forward kinematics of the Franka FR3

We use the dataset collected by the babbling robots (`run_collect`).
If you did not collect it, the pretrained one is used automatically.

Goals:
1. look at the data (always do it before training!);
2. train $x = f_\theta(q)$, $q \in \mathbb{R}^7$, $x \in \mathbb{R}^3$;
3. evaluate it honestly, also on data from a *different* distribution;
4. differentiate the network: the **learned Jacobian**, that our controller will use.

In [ ]:
import os, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from robot_learning_course import fr3
from robot_learning_course.paths import DATA_DIR, MODELS_DIR, find_file
from robot_learning_course.learning.data import load_dataset, split_indices, make_loader
from robot_learning_course.learning.models import MLP, save_model, load_model
from robot_learning_course.kinematics import TrueKinematics

TRAIN = True     # False = skip training and load the pretrained model (fast)
torch.manual_seed(0)

## 1. Look at the data

In [ ]:
q, x = load_dataset(os.path.join(DATA_DIR, "fk_dataset.npz"))
print("q:", tuple(q.shape), " x:", tuple(x.shape))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for j in range(7):
    ax[0].hist(q[:, j].numpy(), bins=50, histtype="step", label=f"joint {j+1}")
ax[0].set(xlabel="angle [rad]", title="joint angles (inputs)"); ax[0].legend(fontsize=7)
ax[1].scatter(x[::10, 0], x[::10, 2], s=1, alpha=0.3)
ax[1].set(xlabel="x [m]", ylabel="z [m]", title="flange positions, side view (labels)"); ax[1].axis("equal")
plt.tight_layout(); plt.show()

❓ *Are the joint angles uniformly distributed? Is the workspace uniformly covered? Where will the model be less accurate?*

## 2. Train
The same code as `train_fk`, step by step.

In [ ]:
train, val, test = split_indices(len(q))
model = MLP(in_dim=7, out_dim=3, hidden=(256, 256, 256), angle_input=True)
model.set_normalization(q[train], x[train])

def error_mm(m, qq, xx):
    with torch.no_grad():
        return 1000 * (m(qq) - xx).norm(dim=1)

if TRAIN:
    loader = make_loader(q[train], x[train], batch_size=256)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    epochs = 60
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    hist = []
    t0 = time.time()
    for epoch in range(epochs):
        for qb, xb in loader:
            loss = (((model(qb) - xb) / model.out_std) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        hist.append((error_mm(model, q[train], x[train]).mean().item(), error_mm(model, q[val], x[val]).mean().item()))
        if epoch % 10 == 0:
            print(f"epoch {epoch:3d}  train {hist[-1][0]:6.2f} mm  val {hist[-1][1]:6.2f} mm  ({time.time()-t0:.0f} s)")
    plt.semilogy(np.array(hist)); plt.legend(["train", "validation"]); plt.xlabel("epoch"); plt.ylabel("mean error [mm]"); plt.grid(True, which="both"); plt.show()
else:
    model = load_model(find_file(os.path.join(MODELS_DIR, "fk_model.pt")))

## 3. Evaluate honestly

In [ ]:
e_test = error_mm(model, q[test], x[test])
print(f"TEST (same distribution): mean {e_test.mean():.2f} mm, 95th percentile {e_test.quantile(0.95):.2f} mm")

# A different distribution: configurations sampled uniformly, positions from the TRUE kinematics
true_kin = TrueKinematics()
rng = np.random.default_rng(42)
q_new = rng.uniform(fr3.JOINT_LOWER, fr3.JOINT_UPPER, size=(5000, 7))
x_new = torch.tensor(true_kin.fk(q_new), dtype=torch.float32)
e_new = error_mm(model, torch.tensor(q_new, dtype=torch.float32), x_new)
print(f"NEW data (uniform q):      mean {e_new.mean():.2f} mm, 95th percentile {e_new.quantile(0.95):.2f} mm")

plt.hist(e_test.numpy(), bins=60, alpha=0.6, density=True, label="test split")
plt.hist(e_new.numpy(), bins=60, alpha=0.6, density=True, label="new uniform data")
plt.xlabel("position error [mm]"); plt.legend(); plt.show()

❓ *Why is the error on new data larger?* Two reasons, both important in practice:
* consecutive babbling samples (10 ms apart) are almost identical: a **random split** puts near-twins in train and test (optional exercise 3);
* the new data covers the joint space differently (**distribution shift**).

## 4. The learned Jacobian
$J_\theta(q) = \partial f_\theta / \partial q$ is a $3\times 7$ matrix: how the flange moves when each joint moves.
PyTorch gives it to us with **one line**. Compare with the true one.

In [ ]:
q0 = torch.tensor(fr3.HOME, dtype=torch.float32)
J_learned = torch.autograd.functional.jacobian(model, q0).numpy()
J_true = true_kin.jacobian(fr3.HOME)
np.set_printoptions(precision=3, suppress=True)
print("learned J:\n", J_learned)
print("true J:\n", J_true)
print(f"relative error: {100 * np.linalg.norm(J_learned - J_true) / np.linalg.norm(J_true):.1f} %")

❓ *Look at the last column. Why is it (almost) zero?* Hint: where is the flange with respect to the axis of joint 7?

### Smoothness matters: sweep one joint and look at a Jacobian entry

In [ ]:
sweep = np.linspace(-1.5, 1.5, 300)
qs = np.tile(fr3.HOME, (300, 1)); qs[:, 1] = sweep
Jx = [torch.autograd.functional.jacobian(model, torch.tensor(qq, dtype=torch.float32))[0, 1].item() for qq in qs]
Jx_true = [true_kin.jacobian(qq)[0, 1] for qq in qs]
plt.plot(sweep, Jx_true, "k", lw=3, label="true")
plt.plot(sweep, Jx, "tab:orange", label="learned (SiLU network)")
plt.xlabel("q2 [rad]"); plt.ylabel("dx / dq2 [m/rad]"); plt.legend(); plt.show()

## ✍️ Optional exercises
* **(2)** In `learning/models.py` replace `nn.SiLU()` with `nn.ReLU()`, retrain, re-run the plot above. What does the Jacobian look like? Why is this bad for a controller?
* **(3)** In `learning/data.py` split by time instead of randomly. How does the test error change?
* Train with 5 000, 20 000, 50 000 samples: plot test error vs dataset size.

In [ ]:
if TRAIN:
    os.makedirs(MODELS_DIR, exist_ok=True)
    save_model(model, os.path.join(MODELS_DIR, "fk_model_notebook.pt"))
    print("saved: use it in the controller with  -p model:=" + os.path.join(MODELS_DIR, "fk_model_notebook.pt"))